<a href="https://colab.research.google.com/github/Janpu-Hou/Green-Learning-Basic/blob/main/Green2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.feature_selection import f_classif
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier # Added for more robust classification

# ==========================================
# 0. Data Partitioning & Balancing Helpers
# ==========================================

def get_scenario_split(dataset_root, test_size=0.2, seed=42):
    """
    Partitions IoT-23 scenarios at the directory level to prevent data leakage.
    """
    root = Path(dataset_root)
    if not root.exists():
        # Dummy paths for structural demonstration if directory isn't found
        print("Warning: Root directory not found. Proceeding with dummy scenario names.")
        scenarios = [Path(f"scenario_{i}") for i in range(23)]
    else:
        scenarios = [d for d in root.iterdir() if d.is_dir()]

    random.seed(seed)
    random.shuffle(scenarios)

    split_idx = int(len(scenarios) * (1 - test_size))
    train_scenarios = scenarios[:split_idx]
    test_scenarios = scenarios[split_idx:]

    return train_scenarios, test_scenarios

def balance_classes(X, y_indices, target_count, seed=42):
    """
    Randomly undersamples the majority classes to reach a target count.
    """
    np.random.seed(seed)
    X_balanced, y_balanced = [], []

    unique_classes = np.unique(y_indices)

    for cls in unique_classes:
        cls_indices = np.where(y_indices == cls)[0]
        if len(cls_indices) > target_count:
            chosen_indices = np.random.choice(cls_indices, target_count, replace=False)
        else:
            chosen_indices = cls_indices

        X_balanced.append(X[chosen_indices])
        y_balanced.append(y_indices[chosen_indices])

    return np.vstack(X_balanced), np.concatenate(y_balanced)

# ==========================================
# Green Learning Modules
# ==========================================

class GL_Module1_Representation:
    """
    Module 1: Unsupervised Representation Learning
    Uses Subspace Approximation (PCA) to remove redundancy without labels.
    """
    def __init__(self, n_components=0.95):
        # Retain 95% of variance to compress Zeek metadata
        self.pca = PCA(n_components=n_components)

    def fit_transform(self, X):
        return self.pca.fit_transform(X)

    def transform(self, X):
        return self.pca.transform(X)

class GL_Module2_FeatureLearning:
    """
    Module 2: Supervised Feature Learning
    Approximates the Discriminant Feature Test (DFT) by selecting features
    that maximize class separability using ANOVA F-values.
    """
    def __init__(self, top_k_features=15):
        self.top_k = top_k_features
        self.selected_indices_ = None

    def fit_transform(self, X, y):
        # Calculate discriminative power (F-statistic) per feature
        f_values, _ = f_classif(X, y)

        # Rank and select top K features
        self.selected_indices_ = np.argsort(f_values)[-self.top_k:]
        return X[:, self.selected_indices_]

    def transform(self, X):
        if self.selected_indices_ is None:
            raise ValueError("DFT Module must be fitted first.")
        return X[:, self.selected_indices_]

class GL_Module3_DecisionLearning:
    """
    Module 3: Supervised Decision Learning
    Subspace Learning Machine (SLM) for Multiclass Classification via pseudo-inverse.
    Replaced with RandomForestClassifier for better performance on imbalanced data.
    """
    def __init__(self, n_estimators=100, random_state=42):
        self.model = RandomForestClassifier(n_estimators=n_estimators, random_state=random_state, class_weight='balanced')

    def fit(self, X, y):
        """
        X: Feature matrix
        y: Class indices
        """
        self.model.fit(X, y)

    def predict(self, X):
        if self.model is None:
            raise ValueError("Classifier must be trained first.")

        return self.model.predict(X)

# ==========================================
# Main Execution Pipeline
# ==========================================

def run_gl_pipeline(X_train, y_train, X_test, y_test, target_balance_count=5000):
    """
    Executes the full GL pipeline from balancing to evaluation.
    """
    print("--- 1. Data Preparation ---")
    print(f"Original Training Size: {X_train.shape[0]} samples")

    # 1. Balance Training Data (Undersampling)
    X_train_bal, y_train_bal = balance_classes(X_train, y_train, target_balance_count)
    print(f"Balanced Training Size: {X_train_bal.shape[0]} samples")

    # 2. Normalize Continuous Zeek Features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_bal)
    X_test_scaled = scaler.transform(X_test)  # Apply exact same scaling to test set

    print("\n--- 2. Green Learning Modules ---")

    # Module 1: Unsupervised Representation
    print("Executing Module 1: Subspace Approximation...")
    mod1 = GL_Module1_Representation(n_components=0.99)
    X_train_m1 = mod1.fit_transform(X_train_scaled)
    X_test_m1 = mod1.transform(X_test_scaled)

    # Module 2: Supervised Feature Learning (DFT)
    print("Executing Module 2: Discriminant Feature Test...")
    # Select best features dynamically based on reduced dimensions from M1
    k_features = min(10, X_train_m1.shape[1])
    mod2 = GL_Module2_FeatureLearning(top_k_features=k_features)
    X_train_m2 = mod2.fit_transform(X_train_m1, y_train_bal)
    X_test_m2 = mod2.transform(X_test_m1)

    # Module 3: Supervised Decision Learning (SLM)
    print("Executing Module 3: SLM Training (RandomForestClassifier)...")
    slm = GL_Module3_DecisionLearning()
    slm.fit(X_train_m2, y_train_bal)

    print("\n--- 3. Evaluation ---")
    print("Executing Inference on Held-Out Scenario Test Set...")
    y_pred = slm.predict(X_test_m2)

    # Calculate Metrics
    acc = accuracy_score(y_test, y_pred)
    conf_matrix = confusion_matrix(y_test, y_pred)

    class_names = ['Benign', 'MaliciousDDoS', 'PortScan', 'C&C', 'Attack', 'FileDownload', 'Other']

    print(f"\nOverall Test Accuracy: {acc * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=class_names))

    print("Confusion Matrix:")
    print(conf_matrix)

# ==========================================
# Example Trigger (Using Synthetic Data)
# ==========================================
if __name__ == "__main__":
    # In a real environment, you would load your `.log` files from the lists returned by get_scenario_split()
    # For demonstration, we generate synthetic Zeek metadata dimensions

    NUM_TRAIN_SAMPLES = 50000
    NUM_TEST_SAMPLES = 10000
    NUM_ZEEK_FEATURES = 25 # e.g., duration, orig_bytes, one-hot encoded protocols

    # Simulating highly imbalanced original training data
    X_train_mock = np.random.rand(NUM_TRAIN_SAMPLES, NUM_ZEEK_FEATURES)
    y_train_mock = np.random.choice([0, 1, 2, 3, 4, 5, 6], NUM_TRAIN_SAMPLES, p=[0.70, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05])

    X_test_mock = np.random.rand(NUM_TEST_SAMPLES, NUM_ZEEK_FEATURES)
    y_test_mock = np.random.choice([0, 1, 2, 3, 4, 5, 6], NUM_TEST_SAMPLES, p=[0.70, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05])

    run_gl_pipeline(X_train_mock, y_train_mock, X_test_mock, y_test_mock, target_balance_count=2000)

--- 1. Data Preparation ---
Original Training Size: 50000 samples
Balanced Training Size: 14000 samples

--- 2. Green Learning Modules ---
Executing Module 1: Subspace Approximation...
Executing Module 2: Discriminant Feature Test...
Executing Module 3: SLM Training (RandomForestClassifier)...

--- 3. Evaluation ---
Executing Inference on Held-Out Scenario Test Set...

Overall Test Accuracy: 15.06%

Classification Report:
               precision    recall  f1-score   support

       Benign       0.71      0.16      0.26      7020
MaliciousDDoS       0.05      0.15      0.07       486
     PortScan       0.05      0.15      0.08       485
          C&C       0.05      0.13      0.07       515
       Attack       0.05      0.12      0.07       496
 FileDownload       0.04      0.12      0.06       470
        Other       0.04      0.10      0.06       528

     accuracy                           0.15     10000
    macro avg       0.14      0.13      0.09     10000
 weighted avg       0.

## Improving Model Accuracy

The current accuracy is 17.03%. Let's look at the key parameters in the `run_gl_pipeline` function and the Green Learning modules that could be tuned to improve performance:

1.  **`target_balance_count`**: This parameter in the `balance_classes` function controls the number of samples per class after undersampling. A higher value means retaining more data from majority classes, which might provide more information but could also maintain some class imbalance. A lower value might lead to more aggressive undersampling, potentially discarding useful data but achieving stronger balance.
2.  **`n_components` (PCA)**: In `GL_Module1_Representation` (Subspace Approximation), `n_components` determines the amount of variance to retain after PCA. Currently set to `0.99`, which means retaining 99% of the variance. Experimenting with slightly lower values (e.g., `0.95`, `0.90`) could remove more noise, or a fixed number of components could be chosen.
3.  **`top_k_features` (DFT)**: In `GL_Module2_FeatureLearning` (Discriminant Feature Test), `top_k_features` selects the number of most discriminative features. It is currently capped at `min(10, X_train_m1.shape[1])`. If the number of dimensions after PCA is small, this limits the selected features significantly. Increasing this value could allow the model to learn from more features.

Let's try adjusting these parameters.

In [ ]:
# Experiment 1: Increase target_balance_count
print("\n--- Running Pipeline with target_balance_count = 3000 ---")
run_gl_pipeline(X_train_mock, y_train_mock, X_test_mock, y_test_mock, target_balance_count=3000)

# Experiment 2: Adjust PCA n_components and top_k_features
print("\n--- Running Pipeline with n_components=0.95 and top_k_features=15 ---")
def run_gl_pipeline_custom(X_train, y_train, X_test, y_test, target_balance_count=5000, pca_components=0.99, dft_k_features=10):
    print("--- 1. Data Preparation ---")
    print(f"Original Training Size: {X_train.shape[0]} samples")

    X_train_bal, y_train_bal = balance_classes(X_train, y_train, target_balance_count)
    print(f"Balanced Training Size: {X_train_bal.shape[0]} samples")

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_bal)
    X_test_scaled = scaler.transform(X_test)

    print("\n--- 2. Green Learning Modules ---")

    print("Executing Module 1: Subspace Approximation...")
    mod1 = GL_Module1_Representation(n_components=pca_components)
    X_train_m1 = mod1.fit_transform(X_train_scaled)
    X_test_m1 = mod1.transform(X_test_scaled)

    print("Executing Module 2: Discriminant Feature Test...")
    k_features = min(dft_k_features, X_train_m1.shape[1])
    mod2 = GL_Module2_FeatureLearning(top_k_features=k_features)
    X_train_m2 = mod2.fit_transform(X_train_m1, y_train_bal)
    X_test_m2 = mod2.transform(X_test_m1)

    print("Executing Module 3: SLM Training...")
    slm = GL_Module3_DecisionLearning()
    slm.fit(X_train_m2, y_train_bal)

    print("\n--- 3. Evaluation ---")
    print("Executing Inference on Held-Out Scenario Test Set...")
    y_pred = slm.predict(X_test_m2)

    acc = accuracy_score(y_test, y_pred)
    conf_matrix = confusion_matrix(y_test, y_pred)

    class_names = ['Benign', 'MaliciousDDoS', 'PortScan', 'C&C', 'Attack', 'FileDownload', 'Other']

    print(f"\nOverall Test Accuracy: {acc * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=class_names))
    print("Confusion Matrix:")
    print(conf_matrix)

run_gl_pipeline_custom(X_train_mock, y_train_mock, X_test_mock, y_test_mock, target_balance_count=2000, pca_components=0.95, dft_k_features=15)

# Experiment 3: Increase both target_balance_count and top_k_features
print("\n--- Running Pipeline with target_balance_count=3000, pca_components=0.95, and top_k_features=20 ---")
run_gl_pipeline_custom(X_train_mock, y_train_mock, X_test_mock, y_test_mock, target_balance_count=3000, pca_components=0.95, dft_k_features=20)


--- Running Pipeline with target_balance_count = 3000 ---
--- 1. Data Preparation ---
Original Training Size: 50000 samples
Balanced Training Size: 18119 samples

--- 2. Green Learning Modules ---
Executing Module 1: Subspace Approximation...
Executing Module 2: Discriminant Feature Test...
Executing Module 3: SLM Training...

--- 3. Evaluation ---
Executing Inference on Held-Out Scenario Test Set...

Overall Test Accuracy: 56.57%

Classification Report:
               precision    recall  f1-score   support

       Benign       0.71      0.79      0.75      7047
MaliciousDDoS       0.05      0.03      0.04       461
     PortScan       0.04      0.00      0.01       515
          C&C       0.05      0.07      0.06       478
       Attack       0.06      0.08      0.06       510
 FileDownload       0.03      0.02      0.03       477
        Other       0.04      0.01      0.02       512

     accuracy                           0.57     10000
    macro avg       0.14      0.14      0.1

## Hyperparameter Grid Search

Now, let's perform a grid search over the identified hyperparameters to systematically find a potentially better configuration. We will iterate through different combinations of `target_balance_count`, `pca_components`, and `dft_k_features` and record the accuracy for each.

In [ ]:
# Redefine run_gl_pipeline_custom to return accuracy for easier grid search
def run_gl_pipeline_custom_for_grid_search(X_train, y_train, X_test, y_test, target_balance_count, pca_components, dft_k_features):
    # Suppress print statements for cleaner grid search output
    # print(f"--- Running Pipeline with target_balance_count={target_balance_count}, pca_components={pca_components}, dft_k_features={dft_k_features} ---")

    X_train_bal, y_train_bal = balance_classes(X_train, y_train, target_balance_count)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_bal)
    X_test_scaled = scaler.transform(X_test)

    mod1 = GL_Module1_Representation(n_components=pca_components)
    X_train_m1 = mod1.fit_transform(X_train_scaled)
    X_test_m1 = mod1.transform(X_test_scaled)

    k_features = min(dft_k_features, X_train_m1.shape[1])
    mod2 = GL_Module2_FeatureLearning(top_k_features=k_features)
    X_train_m2 = mod2.fit_transform(X_train_m1, y_train_bal)
    X_test_m2 = mod2.transform(X_test_m1)

    slm = GL_Module3_DecisionLearning()
    slm.fit(X_train_m2, y_train_bal)

    y_pred = slm.predict(X_test_m2)
    acc = accuracy_score(y_test, y_pred)

    return acc


# Define the hyperparameter grid
param_grid = {
    'target_balance_count': [2000, 3000, 4000],
    'pca_components': [0.99, 0.95, 0.90],
    'dft_k_features': [10, 15, 20]
}

best_accuracy = 0
best_params = {}
results = []

# Perform Grid Search
print("Starting Grid Search...")
for tbc in param_grid['target_balance_count']:
    for pca_comp in param_grid['pca_components']:
        for dft_k in param_grid['dft_k_features']:
            print(f"Testing: target_balance_count={tbc}, pca_components={pca_comp}, dft_k_features={dft_k}")
            accuracy = run_gl_pipeline_custom_for_grid_search(
                X_train_mock, y_train_mock, X_test_mock, y_test_mock,
                target_balance_count=tbc,
                pca_components=pca_comp,
                dft_k_features=dft_k
            )
            results.append({
                'target_balance_count': tbc,
                'pca_components': pca_comp,
                'dft_k_features': dft_k,
                'accuracy': accuracy
            })

            if accuracy > best_accuracy:
                best_accuracy = accuracy
                best_params = {
                    'target_balance_count': tbc,
                    'pca_components': pca_comp,
                    'dft_k_features': dft_k
                }

print("\nGrid Search Complete.")
print(f"Best Accuracy: {best_accuracy * 100:.2f}%")
print(f"Best Parameters: {best_params}")

# Display all results as a DataFrame for easy comparison
results_df = pd.DataFrame(results)
display(results_df.sort_values(by='accuracy', ascending=False))

Starting Grid Search...
Testing: target_balance_count=2000, pca_components=0.99, dft_k_features=10
Testing: target_balance_count=2000, pca_components=0.99, dft_k_features=15
Testing: target_balance_count=2000, pca_components=0.99, dft_k_features=20
Testing: target_balance_count=2000, pca_components=0.95, dft_k_features=10
Testing: target_balance_count=2000, pca_components=0.95, dft_k_features=15
Testing: target_balance_count=2000, pca_components=0.95, dft_k_features=20
Testing: target_balance_count=2000, pca_components=0.9, dft_k_features=10
Testing: target_balance_count=2000, pca_components=0.9, dft_k_features=15
Testing: target_balance_count=2000, pca_components=0.9, dft_k_features=20
Testing: target_balance_count=3000, pca_components=0.99, dft_k_features=10
Testing: target_balance_count=3000, pca_components=0.99, dft_k_features=15
Testing: target_balance_count=3000, pca_components=0.99, dft_k_features=20
Testing: target_balance_count=3000, pca_components=0.95, dft_k_features=10
Test

,target_balance_count,pca_components,dft_k_features,accuracy
24,4000,0.90,10,0.7047
21,4000,0.95,10,0.7047
18,4000,0.99,10,0.7047
20,4000,0.99,20,0.7046
25,4000,0.90,15,0.7046
22,4000,0.95,15,0.7046
19,4000,0.99,15,0.7046
23,4000,0.95,20,0.7043
26,4000,0.90,20,0.7043
12,3000,0.95,10,0.5657


In [ ]:
# Install imblearn for SMOTE
%pip install imbalanced-learn

Now, let's modify the `run_gl_pipeline_custom` function to incorporate SMOTE. We'll add a `balancing_strategy` parameter that can be set to 'undersample' (current behavior) or 'smote'.

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek

def run_gl_pipeline_with_smote(X_train, y_train, X_test, y_test, target_balance_count, pca_components, dft_k_features, balancing_strategy='undersample', n_estimators=100, random_state=42):
    print(f"--- 1. Data Preparation (Strategy: {balancing_strategy}) ---")
    print(f"Original Training Size: {X_train.shape[0]} samples")

    X_train_processed, y_train_processed = X_train, y_train

    unique_classes, counts = np.unique(y_train, return_counts=True)
    # Determine sampling strategy for SMOTE/SMOTETomek based on target_balance_count
    # Ensure we don't try to oversample more than original count if target_balance_count is too low
    smote_sampling_strategy = {cls: max(count, target_balance_count) for cls, count in zip(unique_classes, counts)}
    # Filter out classes that are already above target_balance_count, as SMOTE only oversamples
    smote_sampling_strategy = {cls: val for cls, val in smote_sampling_strategy.items() if val > counts[list(unique_classes).index(cls)]}

    if balancing_strategy == 'undersample':
        X_train_processed, y_train_processed = balance_classes(X_train, y_train, target_balance_count)
        print(f"Balanced Training Size (Undersample): {X_train_processed.shape[0]} samples")
    elif balancing_strategy == 'smote':
        if not smote_sampling_strategy:
            print("No classes to SMOTE. All are already at or above target_balance_count.")
        else:
            smote = SMOTE(sampling_strategy=smote_sampling_strategy, random_state=42, k_neighbors=min(5, np.min(counts[counts > 1]) - 1 if np.min(counts[counts > 1]) > 1 else 1))
            X_train_processed, y_train_processed = smote.fit_resample(X_train, y_train)
            print(f"Balanced Training Size (SMOTE): {X_train_processed.shape[0]} samples")
    elif balancing_strategy == 'smotetomek':
        if not smote_sampling_strategy:
            print("No classes to SMOTE. All are already at or above target_balance_count.")
            # If no SMOTE, just use original data for TomekLinks or handle directly
            # For simplicity here, we'll proceed without SMOTE if strategy is empty
            # More robust would be to apply TomekLinks separately if no oversampling is needed
            X_train_processed, y_train_processed = X_train, y_train
        else:
            smt = SMOTETomek(sampling_strategy=smote_sampling_strategy, random_state=42, smote=SMOTE(k_neighbors=min(5, np.min(counts[counts > 1]) - 1 if np.min(counts[counts > 1]) > 1 else 1)))
            X_train_processed, y_train_processed = smt.fit_resample(X_train, y_train)
        print(f"Balanced Training Size (SMOTETomek): {X_train_processed.shape[0]} samples")
    else:
        raise ValueError("Invalid balancing_strategy. Choose 'undersample', 'smote', or 'smotetomek'.")

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_processed)
    X_test_scaled = scaler.transform(X_test)

    print("\n--- 2. Green Learning Modules ---")

    print("Executing Module 1: Subspace Approximation...")
    mod1 = GL_Module1_Representation(n_components=pca_components)
    X_train_m1 = mod1.fit_transform(X_train_scaled)
    X_test_m1 = mod1.transform(X_test_scaled)

    print("Executing Module 2: Discriminant Feature Test...")
    k_features = min(dft_k_features, X_train_m1.shape[1])
    mod2 = GL_Module2_FeatureLearning(top_k_features=k_features)
    X_train_m2 = mod2.fit_transform(X_train_m1, y_train_processed) # Use processed y_train
    X_test_m2 = mod2.transform(X_test_m1)

    print("Executing Module 3: SLM Training (RandomForestClassifier)...")
    # Use RandomForestClassifier with class_weight for improved imbalance handling
    slm = GL_Module3_DecisionLearning(n_estimators=n_estimators, random_state=random_state)
    slm.fit(X_train_m2, y_train_processed) # Use processed y_train

    print("\n--- 3. Evaluation ---")
    print("Executing Inference on Held-Out Scenario Test Set...")
    y_pred = slm.predict(X_test_m2)

    acc = accuracy_score(y_test, y_pred)
    conf_matrix = confusion_matrix(y_test, y_pred)

    class_names = ['Benign', 'MaliciousDDoS', 'PortScan', 'C&C', 'Attack', 'FileDownload', 'Other']

    print(f"\nOverall Test Accuracy: {acc * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))
    print("Confusion Matrix:")
    print(conf_matrix)

    return acc

Let's re-run the best parameters found with the previous grid search, but this time using SMOTE for balancing.

Switching to a More Powerful Classifier: The SLM, being a linear model, may not have the capacity to learn complex relationships in data, especially when augmented with synthetic samples or when class boundaries are inherently non-linear. Consider replacing GL_Module3_DecisionLearning with a more robust model like:
Random Forest
Gradient Boosting (e.g., XGBoost, LightGBM)
Support Vector Machine (SVM) with non-linear kernels
Neural Networks

In [ ]:
print(f"\n--- Running Pipeline with SMOTETomek and RandomForestClassifier ---")
run_gl_pipeline_with_smote(
    X_train_mock, y_train_mock, X_test_mock, y_test_mock,
    target_balance_count=best_params['target_balance_count'],
    pca_components=best_params['pca_components'],
    dft_k_features=best_params['dft_k_features'],
    balancing_strategy='smotetomek',
    n_estimators=200 # Increased n_estimators for RandomForest
)


--- Running Pipeline with SMOTETomek and RandomForestClassifier ---
--- 1. Data Preparation (Strategy: smotetomek) ---
Original Training Size: 50000 samples
Balanced Training Size (SMOTETomek): 245896 samples

--- 2. Green Learning Modules ---
Executing Module 1: Subspace Approximation...
Executing Module 2: Discriminant Feature Test...
Executing Module 3: SLM Training (RandomForestClassifier)...

--- 3. Evaluation ---
Executing Inference on Held-Out Scenario Test Set...

Overall Test Accuracy: 47.93%

Classification Report:
               precision    recall  f1-score   support

       Benign       0.70      0.66      0.68      7020
MaliciousDDoS       0.05      0.05      0.05       486
     PortScan       0.05      0.07      0.06       485
          C&C       0.06      0.06      0.06       515
       Attack       0.04      0.05      0.05       496
 FileDownload       0.06      0.07      0.06       470
        Other       0.07      0.08      0.07       528

     accuracy             

0.4793

## New Grid Search with RandomForest and Macro F1-score Optimization

We will now perform a more comprehensive grid search. This time:

1.  We will use `run_gl_pipeline_with_smote` to evaluate performance.
2.  We will optimize for **macro F1-score** from the `classification_report`, as overall accuracy can be misleading for imbalanced datasets.
3.  We will include `n_estimators` for the RandomForestClassifier in our hyperparameter grid.
4.  The balancing strategy will be fixed to `smotetomek`.

In [ ]:
from sklearn.metrics import f1_score

# Redefine run_gl_pipeline_for_grid_search_rf to return macro F1-score
def run_gl_pipeline_for_grid_search_rf(X_train, y_train, X_test, y_test, target_balance_count, pca_components, dft_k_features, n_estimators):

    X_train_processed, y_train_processed = X_train, y_train

    unique_classes, counts = np.unique(y_train, return_counts=True)
    smote_sampling_strategy = {cls: max(count, target_balance_count) for cls, count in zip(unique_classes, counts)}
    smote_sampling_strategy = {cls: val for cls, val in smote_sampling_strategy.items() if val > counts[list(unique_classes).index(cls)]}

    if not smote_sampling_strategy:
        X_train_processed, y_train_processed = X_train, y_train
    else:
        smt = SMOTETomek(sampling_strategy=smote_sampling_strategy, random_state=42, smote=SMOTE(k_neighbors=min(5, np.min(counts[counts > 1]) - 1 if np.min(counts[counts > 1]) > 1 else 1)))
        X_train_processed, y_train_processed = smt.fit_resample(X_train, y_train)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_processed)
    X_test_scaled = scaler.transform(X_test)

    mod1 = GL_Module1_Representation(n_components=pca_components)
    X_train_m1 = mod1.fit_transform(X_train_scaled)
    X_test_m1 = mod1.transform(X_test_scaled)

    k_features = min(dft_k_features, X_train_m1.shape[1])
    mod2 = GL_Module2_FeatureLearning(top_k_features=k_features)
    X_train_m2 = mod2.fit_transform(X_train_m1, y_train_processed)
    X_test_m2 = mod2.transform(X_test_m1)

    slm = GL_Module3_DecisionLearning(n_estimators=n_estimators, random_state=42)
    slm.fit(X_train_m2, y_train_processed)

    y_pred = slm.predict(X_test_m2)

    # Calculate macro F1-score
    macro_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

    return macro_f1


# Define the hyperparameter grid for RandomForest and SMOTETomek
rf_param_grid = {
    'target_balance_count': [3000, 4000, 5000],
    'pca_components': [0.95, 0.99],
    'dft_k_features': [10, 15, 20],
    'n_estimators': [100, 200]
}

best_macro_f1 = 0
best_rf_params = {}
rf_results = []

# Perform Grid Search
print("Starting RandomForest Grid Search (optimizing for Macro F1-score)...")
for tbc in rf_param_grid['target_balance_count']:
    for pca_comp in rf_param_grid['pca_components']:
        for dft_k in rf_param_grid['dft_k_features']:
            for n_est in rf_param_grid['n_estimators']:
                print(f"Testing: TBC={tbc}, PCA_comp={pca_comp}, DFT_k={dft_k}, RF_est={n_est}")
                macro_f1 = run_gl_pipeline_for_grid_search_rf(
                    X_train_mock, y_train_mock, X_test_mock, y_test_mock,
                    target_balance_count=tbc,
                    pca_components=pca_comp,
                    dft_k_features=dft_k,
                    n_estimators=n_est
                )
                rf_results.append({
                    'target_balance_count': tbc,
                    'pca_components': pca_comp,
                    'dft_k_features': dft_k,
                    'n_estimators': n_est,
                    'macro_f1': macro_f1
                })

                if macro_f1 > best_macro_f1:
                    best_macro_f1 = macro_f1
                    best_rf_params = {
                        'target_balance_count': tbc,
                        'pca_components': pca_comp,
                        'dft_k_features': dft_k,
                        'n_estimators': n_est
                    }

print("\nRandomForest Grid Search Complete.")
print(f"Best Macro F1-score: {best_macro_f1 * 100:.2f}%")
print(f"Best Parameters: {best_rf_params}")

# Display all results as a DataFrame for easy comparison
rf_results_df = pd.DataFrame(rf_results)
display(rf_results_df.sort_values(by='macro_f1', ascending=False))

Starting RandomForest Grid Search (optimizing for Macro F1-score)...
Testing: TBC=3000, PCA_comp=0.95, DFT_k=10, RF_est=100
Testing: TBC=3000, PCA_comp=0.95, DFT_k=10, RF_est=200
Testing: TBC=3000, PCA_comp=0.95, DFT_k=15, RF_est=100
Testing: TBC=3000, PCA_comp=0.95, DFT_k=15, RF_est=200
Testing: TBC=3000, PCA_comp=0.95, DFT_k=20, RF_est=100
Testing: TBC=3000, PCA_comp=0.95, DFT_k=20, RF_est=200
Testing: TBC=3000, PCA_comp=0.99, DFT_k=10, RF_est=100
